# Absolute calibration of an ARM C-band radar: four independent references**Reflectivity ($Z$) and differential reflectivity ($Z_{DR}$) calibration of theARM C-SAPR2 at Bankhead National Forest (BNF), using clutter, disdrometers, aground radar and a spaceborne radar.**A radar's absolute calibration cannot be established from the radar alone. Thisnotebook works through four references that fail in *different* ways, so thatagreement between them is informative:| Reference | What it constrains | Independent of || --- | --- | --- || **RCA** — relative calibration adjustment from ground clutter | *Changes* in receiver gain, day to day | Precipitation, other instruments || **QUANTS** — ARM disdrometer DSD forward-modelled to C band | Absolute $Z$ and $Z_{DR}$ at a point | Radar hardware, satellites || **NEXRAD** — nearby S-band ground radar (KHTX) | Absolute $Z$ over a large overlap | C-band hardware, spaceborne cal || **GPM DPR** — spaceborne Ku radar, SR–GR volume matching | Absolute $Z$ aloft, globally traceable | Everything ground-based |RCA answers *"did it move?"*; the other three answer *"where is it?"*. You needboth: a stable radar can be stably wrong.## The acceptance test used throughoutA receiver-gain error is **multiplicative in linear power**, therefore**constant in decibels**. Any estimator claiming to measure one must show:1. high correlation with the reference,2. a regression slope consistent with 1 (orthogonal/Deming, not OLS),3. **no dependence of the difference on intensity**, and4. no dependence on range.An estimate that fails these is not a calibration offset, however clean itserror bar. This notebook reports failures as results.

## Prerequisites| Concept | Importance | Notes || --- | --- | --- || [Py-ART](https://arm-doe.github.io/pyart/) | Necessary | Radar I/O and geometry || [ACT](https://arm-doe.github.io/ACT/) | Necessary | ARM data discovery and download || [Radar cross-comparison](https://openradarscience.org/erad2026/) | Helpful | This notebook follows the ERAD 2026 SR–GR notebook || Dual-polarisation basics | Helpful | $Z_{DR}$, $\Phi_{DP}$, $K_{DP}$ |**Time to run:** ~15 min for the single-case walkthrough below.**Do not** run the bulk loops as written — see the caution immediately below.- **ARM account** with an [ARM Live](https://adc.arm.gov/armlive/) token- **NASA Earthdata** login for GPM (or PPS credentials for `gpm-api`)

:::{admonition} **Caution: ARM Live is a streaming interface, not a bulk one**:class: warning**Every cell below that downloads radar volumes will be slow, and will getslower the more you ask for.** C-SAPR2 PPI volumes are **270–720 MB each**.Measured throughput on ARM Live during development ranged from **3.4 MB/s downto 0.30 MB/s**, i.e. **2 to 40 minutes per volume**, and sustained requestsreturned `IncompleteRead` and `502 Bad Gateway` before the service stoppedlisting files altogether.**For anything beyond a handful of volumes, place a data order instead.**Ordering is a session-authenticated action in[ARM Data Discovery](https://adc.arm.gov/discovery/); the resulting order isthen retrievable non-interactively. **ARM Live is for probing and for singlecases — treat a multi-day request through it as a mistake, not a slow success.**Consequences to design around:- **Write every loop restart-safe.** Skip files already on disk, checkpoint  per-file results, and never hold a pass open across a laptop sleep.- **A short file is a torn download, not a small volume.** Check sizes against  a floor before trusting a file.- **Delete granules after extracting what you need.** The GPM screen below  keeps a few variables per granule rather than the 200 MB file.:::

## Imports

In [ ]:
import osimport actimport gpmimport gpm.gv as gvimport matplotlib.pyplot as pltimport numpy as npimport pyartimport xradar  # noqa: F401  -- cfradial readers used via the Py-ART bridgeimport cmweather  # noqa: F401  -- registers ChaseSpectral and friendsDATASTREAM = "bnfcsapr2cfrS3.a1"ARM_USER, ARM_TOKEN = os.environ["ARMUSER"], os.environ["ARMTOKEN"]

### Let ACT tell you where the radar isHard-coding site coordinates is a silent-failure risk: a redeployment moves thepad and the number in your notebook keeps working. `act.utils` queries ARM'sown facility database instead.

In [ ]:
# Returns {"<site> <facility>": {"latitude": ..., "longitude": ...}}. Note the# key echoes the site code in the case you PASSED it, not upper case -- take the# single value rather than indexing by a guessed key.site = act.utils.arm_site_location_search(site_code="bnf", facility_code="S3")(coords,) = site.values()RADAR_LAT, RADAR_LON = coords["latitude"], coords["longitude"]print(f"C-SAPR2 at {RADAR_LAT:.4f}, {RADAR_LON:.4f}")   # 34.6308, -87.1331

:::{admonition} Know your epoch before pooling anything:class: important**Receiver gain does not survive a redeployment.** The BNF record splits intoseveral epochs separated by long downtimes, and they do **not** share acalibration — processor version, range-gate geometry and ray count all changeacross a return-to-service, while the site coordinates stay identical.**This notebook is scoped to a single epoch** (`sapr2cfr-1.17.0`, June 2026onward, 1050 gates × 108 m, 13 952 rays per volume). Every number below appliesto that epoch alone. Pooling epochs into one offset hides the very thing worthmeasuring.A useful control from the disdrometer record: a 210-day downtime changedcalibration by **+0.001 dB** (*p* = 0.68). **Downtime alone does notrecalibrate a radar; a return-to-service can.** So check the processor versionand scan geometry before assuming two files share a calibration.:::

## Two file-level traps in the ARM radar archiveBoth of these produce plausible-looking numbers rather than errors.**1. Volumes are split at UTC midnight.** The first file of each day is a*truncated half*: its declared ray count is the full volume's, while the arrayholds a fraction of it. Reading "sweep 0" by index from the second half returnsa high-elevation beam, not the lowest tilt. Always check actual array boundsagainst `sweep_start_ray_index` / `sweep_end_ray_index`.**2. `uncorrected_reflectivity_h` is unthresholded.** No-signal gates carrygarbage — Doppler pegged at Nyquist, spectral width saturated, coherent powernear zero — and a naive mean will happily "find" 60 dBZ of noise. RequireSNR and coherent power before trusting it.Use `uncorrected_reflectivity_h` for clutter work (the processor-applied fieldsuppresses ground clutter by ~12 dB) and the corrected field for precipitation.

In [ ]:
def quality_filter(radar, snr_min=10.0, ncp_min=0.3, rhohv_min=0.7):    """Build a Py-ART GateFilter for trustworthy precipitation gates.    Parameters    ----------    radar : pyart.core.Radar        Volume to filter.    snr_min : float, optional        Minimum copolar signal-to-noise ratio, dB.    ncp_min : float, optional        Minimum normalized coherent power, unitless.    rhohv_min : float, optional        Minimum copolar correlation coefficient, unitless.    Returns    -------    pyart.filters.GateFilter        Filter accepted directly by Py-ART's correction and retrieval        routines, so the same gate selection is reused downstream rather        than re-derived per call site.    """    gf = pyart.filters.GateFilter(radar)    gf.exclude_below("signal_to_noise_ratio_copolar_h", snr_min)    gf.exclude_below("normalized_coherent_power", ncp_min)    gf.exclude_below("copol_correlation_coeff", rhohv_min)    gf.exclude_invalid("uncorrected_reflectivity_h")    gf.exclude_masked("uncorrected_reflectivity_h")    return gfdef sweep_bounds(radar, sweep=0, field="uncorrected_reflectivity_h"):    """Ray bounds of one sweep, clipped to the array that is actually present.    Parameters    ----------    radar : pyart.core.Radar        Volume to index.    sweep : int, optional        Sweep index.    field : str, optional        Field whose first dimension sets the clip.    Returns    -------    tuple of int        ``(lo, hi)`` for slicing rays.    Notes    -----    ARM splits volumes at UTC midnight, so the declared ray count can exceed    the array present in a truncated half-file. Clipping against the real    shape is what keeps "sweep 0" from silently becoming a high tilt.    """    lo = int(radar.sweep_start_ray_index["data"][sweep])    hi = int(radar.sweep_end_ray_index["data"][sweep]) + 1    return lo, min(hi, radar.fields[field]["data"].shape[0])

## 1. RCA — is the receiver stable?The relative calibration adjustment (Silberstein et al. 2008) tracks the returnfrom **ground clutter**, which is a fixed target: any change in its apparentreflectivity is the radar changing, not the world. It gives no absolutereference, but it tells you whether an absolute offset measured on one dayapplies to another.The statistic below is the mean of the strongest clutter gates in the lowesttilt, per volume, differenced against a baseline. Taking the *top-N gates*rather than a fixed mask makes it insensitive to how the clutter mask was drawn.

In [ ]:
def rca_statistic(radar, n_gates=15, max_range_km=10.0):    """Per-volume clutter-return statistic for RCA.    Parameters    ----------    radar : pyart.core.Radar        Volume; the lowest sweep is used.    n_gates : int, optional        Number of strongest clutter gates to average.    max_range_km : float, optional        Range ceiling for clutter, km.    Returns    -------    float        Mean reflectivity of the strongest clutter gates, dBZ.    """    # pyart.util.subset_radar gives a real Radar restricted to the clutter    # annulus and lowest tilt, so the statistic is computed on a volume whose    # geometry is self-describing rather than on hand-sliced arrays.    near = pyart.util.subset_radar(        radar,        ["uncorrected_reflectivity_h"],        rng_min=0.0,        rng_max=max_range_km * 1e3,        ele_min=0.0,        ele_max=2.0,    )    block = np.ma.asarray(        near.fields["uncorrected_reflectivity_h"]["data"]    ).compressed()    if block.size < n_gates:        return np.nan    return float(np.mean(np.sort(block)[-n_gates:]))

Py-ART's display classes make the clutter field itself easy to inspect — whichis worth doing once before trusting any RCA series, because the statistic isonly as good as the assumption that those gates are ground targets.

In [ ]:
radar = pyart.io.read_cfradial(clutter_volume)          # a dry-day volumegf = quality_filter(radar)display = pyart.graph.RadarMapDisplay(radar)fig, ax = plt.subplots(figsize=(5.5, 5.0),                       subplot_kw={"projection": display.grid_projection})display.plot_ppi_map(    "uncorrected_reflectivity_h",    sweep=0,    ax=ax,    vmin=-10,    vmax=70,    cmap="ChaseSpectral",          # registered by cmweather    colorbar_label="Uncorrected $Z_H$ [dBZ]",    min_lon=RADAR_LON - 0.4,    max_lon=RADAR_LON + 0.4,    min_lat=RADAR_LAT - 0.4,    max_lat=RADAR_LAT + 0.4,)display.plot_range_rings([5, 10], ax=ax, col="0.4", ls="--", lw=0.6)ax.set_title("Ground clutter within 10 km — the RCA target")

:::{admonition} RCA result:class: tip731 volumes over five days: baseline **60.83 dBZ**, scatter **0.349 dB**(MAD-$\sigma$), no significant drift. The receiver is stable at the third-of-a-dBlevel, so a single well-measured absolute offset is transferable across theepoch.:::

:::{admonition} What RCA established, and what it did not:class: tip**Stable to 0.349 dB** (MAD-$\sigma$) across 731 volumes, no significant drift.That licenses transferring one absolute offset across the epoch.It also *corrected* an earlier misdiagnosis: an apparent 45–70 dB collapse inclutter return at 00:00 UTC looked like a reduced-power startup state and wasin fact the midnight file split described above. **A dramatic instrument"finding" at a round clock time deserves suspicion before interpretation.**:::

## 2. QUANTS — an absolute reference at a pointARM's `LDQUANTS` / `VDISQUANTS` products forward-model the measured drop sizedistribution to reflectivity **at radar wavelengths**, including C band. Thatmakes them a true absolute reference, independent of any radar.Two gotchas that cost real time:- `num_concen` is **not** the plain gamma intercept. Only the normalised-gamma  form, using `norm_num_concen` together with `med_diameter`, reproduces ARM's  own published T-matrix $Z$.- The comparison is **one gate against one instrument**: a 942 m vertical  separation plus advection, in a field whose across-scene standard deviation  is 8.5 dB. Small samples are dominated by that mismatch, not by calibration.

In [ ]:
files = act.discovery.download_arm_data(    ARM_USER, ARM_TOKEN, "bnfldquantsM1.c1",    "2026-08-01", "2026-08-06", output="./quants",)ldq = act.io.arm.read_arm_netcdf(files)# ACT reads ARM's embedded QC bit fields and can also pull the Data Quality# Reports filed against this datastream -- a DQR is a human judgement that no# amount of local screening will reproduce.ldq = act.qc.arm.add_dqr_to_qc(ldq, variable="reflectivity_factor_cband20c")# QCFilter turns the bit fields into a masked variable, so downstream code# cannot silently use a flagged sample.z_c = ldq.qcfilter.get_masked_data(    "reflectivity_factor_cband20c", rm_assessments=["Bad", "Incorrect", "Suspect"])zdr_c = ldq.qcfilter.get_masked_data(    "differential_reflectivity_cband20c", rm_assessments=["Bad", "Incorrect"])print(f"{int(z_c.count())} of {z_c.size} samples survive QC")

`act.plotting.DistributionDisplay` is the quick way to see whether thedisdrometer and radar populations even overlap before comparing their medians —if they do not, no offset computed from them means anything.

In [ ]:
# DistributionDisplay offers plot_stairstep / plot_violin / plot_scatter --# there is no plot_histogram, so reach for stairstep for a 1-D distribution.disp = act.plotting.DistributionDisplay({"quants": ldq}, figsize=(5.2, 3.4))disp.plot_stairstep("reflectivity_factor_cband20c", dsname="quants",                    bins=np.arange(0, 55, 2))disp.axes[0].set_xlabel("DSD forward-modelled C-band $Z$ [dBZ]")

:::{admonition} QUANTS results, and a sample-size warning:class: tipOver a full census of the epoch (73 screened rain pairs) the disdrometercomparison gives a median $Z$ offset of **−8.04 dB**. $Z_{DR}$ in light raincame out **−0.27 ± 0.13 dB** (*n* = 7).**But a six-day slice of the same comparison is not enough.** Restricted to2026-08-01…06, only **113 of 8 621** one-minute samples survive qualityscreening, and that subsample does not support a defensible offset. Worse, afirst pass on it gave a convincing **−6.1 ± 1.3 dB** that *inverted* when thequality filter was corrected. **A quality filter can reverse your conclusion;derive the screen before looking at the answer.**:::

## 3. NEXRAD — an S-band ground referenceThe nearby WSR-88D (KHTX, ~72 km away) sees the same storms at a wavelengthwhere attenuation is negligible. Py-ART's `columnsect` extracts verticalcolumns over a location from each radar, which are then matched.Two design decisions dominate the result.

In [ ]:
# Pair each C-SAPR2 volume with its OWN nearest NEXRAD scan.## Anchoring every case to one satellite-crossing time instead leaves tens of# minutes between the two radars; correlation was positive at short offset and# NEGATIVE at long offset -- the two radars were simply looking at different# rain. Per-volume pairing costs almost nothing (NEXRAD volumes are ~6 MB) and# brought median |offset| to 1.3 min.csapr = pyart.io.read_cfradial(csapr_file)nexrad = pyart.io.read_nexrad_archive(nexrad_file)# pyart.util.columnsect extracts the vertical column over a point from each# radar and interpolates to a common height axis -- the same helper both sides.col_c = pyart.util.columnsect.get_field_location(csapr, site_lat, site_lon)col_k = pyart.util.columnsect.get_field_location(nexrad, site_lat, site_lon)

### Estimate attenuation with Py-ART rather than by handWhether the difference is gain or path loss is the central ambiguity in aC-band comparison. Py-ART has both pieces: a $K_{DP}$ retrieval and theZ-PHI attenuation routine of Gu et al., both of which accept the `GateFilter`built earlier so the gate selection stays consistent.$\Phi_{DP}$ at this radar is **uncalibrated** (median ~256°) and in the earlierdeployment **wrapped**, so it can never be used as an absolute path integral.$K_{DP}$ is its range derivative, which makes a constant system offsetdifferentiate away — so retrieve $K_{DP}$ and integrate that instead.

In [ ]:
kdp, phidp_filt, _ = pyart.retrieve.kdp_maesaka(csapr, gatefilter=gf,                                                psidp_field="uncorrected_differential_phase")csapr.add_field("specific_differential_phase", kdp, replace_existing=True)spec_at, pia, cor_z, _, _, _ = pyart.correct.calculate_attenuation_zphi(    csapr,    gatefilter=gf,    refl_field="uncorrected_reflectivity_h",    phidp_field="uncorrected_differential_phase",    zdr_field="uncorrected_differential_reflectivity",    fzl=melting_level_m,          # from a co-located sounding)csapr.add_field("path_integrated_attenuation", pia, replace_existing=True)

:::{admonition} Test attenuation with reflectivity controlled for:class: warningRegressing the difference on a path-attenuation estimate looks compelling andis easy to get wrong: DPR's own `piaFinal` correlates **+0.79 withreflectivity**, so a naive regression re-expresses the intensity slope as atextbook attenuation signature. Control for $Z$ and the dependence vanished(partial slope +0.07, *p* = 0.38).The diagnostic that *does* discriminate: a real path loss gives a slope of**−1 dB per dB** of two-way PIA. Anything much shallower is not extinction.:::

### The unit of comparison decides whether this works at allAt **single-column** scale the comparison fails its acceptance test(correlation 0.71 against a 0.80 bar) — and the limiter is neither raincoverage nor sample size. Subsampling well-filled volumes leaves correlation**flat at 0.75 from *n* = 467 to *n* = 4 674**: more data could never havefixed it. Per-column mismatch noise is **5.6 dB per radar** against a 9.0 dBlatent signal.Averaging columns **in linear power** into 11 km blocks removes it:| Averaging scale | Correlation || --- | --- || 1 km (column) | 0.711 || 5 km | 0.815 || **11 km** | **0.857** || 22 km | 0.847 |Correlation *saturates* around 11 km, which is why that scale is chosen ratherthan tuned. With blocks and a requirement that **both** radars sit >10 dB abovetheir own noise floors, all four acceptance criteria pass.

:::{admonition} A selection trap that manufactures ~8 dB from nothing:class: warningRestricting the comparison to a **reflectivity window on the reference**(e.g. 12–35 dBZ) over a population that is mostly rain-free produces a largeapparent deficit from a **perfectly calibrated** radar. Under a latent-truthmodel — both radars observing a common field with independent errors — a trueoffset of **exactly 0 dB** returns a windowed estimate of **−7.8 dB**.The mechanism is selecting on a *noisy* reference: the window preferentiallyadmits reference gates scattered upward, so the test radar looks low byconstruction. Apply the window to **both** sides or to **neither**, and alwaysrun the null test: **feed a known 0 dB offset through your estimator andconfirm it returns 0.** The block estimator above returns −0.07 to −0.21 dB.:::

## 4. GPM DPR — a spaceborne referenceSpaceborne–ground radar (SR–GR) volume matching after Schwaller & Morris (2011)and Warren et al. (2018), following the[ERAD 2026 SR–GR notebook](https://openradarscience.org/erad2026/notebooks/gpm-api/sr-gr-calibration/).DPR Ku is the most globally traceable reference available, at the cost ofmatching across very different geometries.

In [ ]:
def dpr_overpass_time(ds, lat0, lon0):    """Time and distance of a DPR granule's closest approach to a site.    Parameters    ----------    ds : xarray.Dataset        Granule opened with ``gpm.open_granule_dataset(..., scan_mode="FS")``.    lat0, lon0 : float        Site latitude and longitude, degrees.    Returns    -------    tuple        ``(time, distance_km)`` at closest approach.    Notes    -----    ``lat``/``lon`` are shaped ``(cross_track, along_track)`` while ``time`` has    length ``along_track`` only. ``unravel_index`` returns ``(i, j)`` = (cross,    along), so the overpass time is ``time[j]`` -- **never** ``time[i]``. Using    ``time[i]`` gave 19:21 UTC instead of 19:54 UTC for one BNF pass, a 33 min    error that looks plausible because it lands inside the granule window.    """    lat, lon = ds["lat"].values, ds["lon"].values    dist = np.hypot((lat - lat0) * 111.0,                    (lon - lon0) * 111.0 * np.cos(np.radians(lat0)))    i, j = np.unravel_index(np.nanargmin(dist), dist.shape)    return ds["time"].values[j], float(dist[i, j])

### Screen the swath, not the rain gaugesTo decide which overpasses are worth fetching radar volumes for, screen on theDPR's **own** precipitation flags inside the radar domain. Surface gauges arethe wrong detector: on one BNF overpass the DPR saw **164 precipitatingfootprints** in the domain while **every** BNF gauge read exactly zero. Pointgauges sample a vanishing fraction of a 113 km domain — the rain was in theradar's view but not over the instruments.Of 20 granules containing the site over 15 Jun – 25 Aug 2026, **10 had rain inthe domain**, and 5 of those had radar volumes within ±30 min.:::{admonition} More footprints is not a better case:class: cautionThe overpass with the **most** precipitating footprints (585) was among the*least* usable: that day's scan strategy reached only **58 km**, so ~84% of itsfootprints had no radar to match against. **Check the scan strategy's maximumrange before ranking cases by footprint count.**:::

### Use `gpm.gv.volume_matching` — do not hand-roll the matchgpm-api ships the Schwaller & Morris geometry as a single routine. It handlesthe parallax correction, the beam-volume intersection, per-footprintaggregation in **linear** power, and — importantly — returns the diagnosticcolumns you need to screen on, including `GR_fraction_covered_area` and the`SR_fraction_*` quality fractions.A hand-rolled version of this got the melting-level filter backwards andreturned a confident, wrong answer (see the caution below). **Prefer thelibrary.**

In [ ]:
# gv.volume_matching wants the GR sweep as an xradar Dataset. Py-ART's xradar# bridge converts an existing Radar without leaving the notebook.dtree = pyart.io.read_cfradial(csapr_file).to_xradar()ds_gr = dtree["sweep_0"].to_dataset()ds_gr = ds_gr.rename({"uncorrected_reflectivity_h": "DBZH"})ds_sr = gpm.open_granule_dataset(granule, scan_mode="FS")df = gv.volume_matching(    ds_gr=ds_gr,    ds_sr=ds_sr,    radar_band="C",              # C-SAPR2; DPR is Ku, converted internally    z_variable_gr="DBZH",    beamwidth_gr=1.0,    z_min_threshold_gr=0,    z_min_threshold_sr=10,    max_gr_range=113_000,        # this epoch's unambiguous range    download_sr=False,           # granule already on disk    display_quicklook=True,)print(df.columns.tolist())

The returned frame carries `SR_zFactorFinal_Ku_mean`, `GR_Z_mean`, the coveragefraction, and per-footprint fractions for clutter, hail, melting layer andno-precipitation. **Screen on those rather than inventing your ownthresholds** — two matter most:- **`GR_fraction_covered_area`** — the fill fraction. Low-fill matches are  mostly no-signal gates compared against a footprint the satellite called  precipitating, and they were the single largest source of spurious offset  here. A floor near 0.7 lifted correlation substantially.- **`SR_fraction_melting_layer`** and **`SR_fraction_hail`** — exclude both;  mixed phase breaks the Ku↔C band conversion.

In [ ]:
ok = (    (df["GR_fraction_covered_area"] > 0.70)    & (df["SR_fraction_no_precip"] < 0.10)    & (df["SR_fraction_clutter"] < 0.10)    & (df["SR_fraction_melting_layer"] < 0.10)    & (df["SR_fraction_hail"] < 0.10))matched = df[ok].copy()matched["diff_dbz"] = matched["GR_Z_mean"] - matched["SR_zFactorFinal_Ku_mean"]print(f"{len(matched)} of {len(df)} matches survive screening; "      f"median offset {matched.diff_dbz.median():+.2f} dB")

gpm-api also provides the diagnostic plots for this comparison, so the summaryfigure is one call rather than a bespoke panel:

In [ ]:
gv.calibration_summary(    matched,    gr_z_column="GR_Z_mean",    sr_z_column="SR_zFactorFinal_Ku_mean",    hue_column="GR_range_mean",    gr_range=(10, 50),    sr_range=(10, 50),)

:::{admonition} If you do work with the profiles directly:class: cautionDPR range bins run **top-down**: bin 0 is ~21.9 km altitude and the last bin isthe surface. So slicing "up to the clutter-free bottom" spans the whole column,and a filter written as *above the melting level* keeps **ice**, not rain.The rain window is **below** the freezing level, with a bright-band margin:```pythonmelt = float(h0[i, j])                       # heightZeroDeg, mok = (np.isfinite(prof)      & (prof > 0)      & (hprof < melt - BB_MARGIN_M)         # below the bright band      & (hprof > 0.0))```Written the other way round this sampled ice at 6.7 km and returned a confident**−8.5 dB**. gpm-api's slicing helpers avoid the whole class of error:`gpm.slice_range_at_bin(bins="binClutterFreeBottom")`,`gpm.slice_range_at_height(h)`,`gpm.slice_range_at_temperature(temperature=275.15)` and`gpm.get_height_at_bin(...)`. **Use them instead of index arithmetic.**:::

:::{admonition} Why this matters more than it sounds:class: warningWritten the other way round, the filter sampled ice at 6.7 km and returned**−8.5 dB** — a number that happened to sit very close to an independentlyexpected value, and so was *easy to believe*. Ice is intrinsically weaker thanrain; the deficit was manufactured by the filter.**A result that matches your prior is not thereby validated.** It earns thesame scrutiny as a surprising one.:::

### Regression dilution masquerading as attenuationAfter matching, a residual dependence of the difference on intensity lookedlike C-band attenuation. It was **regression dilution**: the apparent slopetracked $r-1$ exactly, which is what noise in the *reference* variableproduces. Extrapolating to simultaneity drove the slope to zero and correlationtoward 0.92.A related trap: DPR's own `piaFinal` correlates **+0.79 with reflectivity**, soregressing the difference on it appears to show a textbook attenuationsignature. **Control for reflectivity** and the dependence vanishes(partial slope +0.07, *p* = 0.38).

## ResultsFour references on one epoch. Every value below passed the acceptance testunless marked otherwise.| Reference | Result | Notes || --- | --- | --- || **RCA** (clutter) | stable to **0.349 dB** | 731 volumes, no significant drift || **QUANTS** (disdrometer) | **−8.04 dB** | *n* = 73 screened rain pairs || **NEXRAD** (KHTX S-band) | **−7.0 to −8.2 dB** | 11 km blocks; range spans the censoring correction || **GPM DPR Ku** | **−5.64 ± 1.37 dB** | *n* = 328 matched footprints, 4 overpasses |**A practical correction is ≈ −7 dB, uncertainty ±1.5 dB.** The spread*between* passing estimators (2.6 dB) exceeds every individual statisticalerror (0.4–1.4 dB), so quoting any single method's error bar would understatethe real uncertainty. **When methods disagree by more than they claim toresolve, report the range.**:::{admonition} The methods do not all measure the same quantity:class: warningBefore combining any two of these, check what each includes:- The **NEXRAD** and **GPM** estimates have path attenuation *excluded* —  tested and found absent at 9–13% of prediction (*p* > 0.5) and slope +0.11  respectively. They are close to pure **gain** terms.- The **QUANTS** figure is a **total deficit**, which a separate census  decomposed into a ~1.9 dB processing-constant step *plus* an  intensity-dependent, attenuation-like residual growing across intensity  bands.So a total deficit and a gain term agreeing closely is **not** independentcorroboration — it can be coincidence. Removing the processing step from theNEXRAD figure leaves ≈ −6.3 dB, which sits **0.66 dB** from the GPM value,inside 1σ. **The attenuation-free routes cluster nearer −6 dB than −8 dB.**:::**An open question worth stating.** Two independent routes here find pathattenuation *absent*, yet an attenuation-like intensity dependence appears inthe disdrometer decomposition for the same epoch. The comparisons differ ingeometry — 11 km blocks at 1.5–3.5 km altitude versus single-gate paths of33–37 km at low elevation — which may be sufficient explanation, but it is notdemonstrated. **Published here as unresolved.**$Z_{DR}$: the light-rain gate estimate and the QUANTS estimate sit **1.5σ**apart (−0.003 ± 0.129 dB vs −0.27 ± 0.13 dB, combined σ 0.183 dB) — consistentin sign and formally compatible, not yet reconciled in magnitude.

## SummaryFour references, chosen because they fail differently. RCA showed the receiverwas stable, licensing a transferable offset; QUANTS, NEXRAD and GPM located it.**Almost none of this needed bespoke code.** `gpm.gv.volume_matching` performsthe entire spaceborne–ground match and returns the screening columns;`pyart.filters.GateFilter` carries one gate selection through every retrieval;`pyart.retrieve.kdp_maesaka` and `pyart.correct.calculate_attenuation_zphi`supply the attenuation diagnostic; `act.qc` applies ARM's own quality bits andData Quality Reports. The hand-rolled parts of this analysis were where themistakes happened.The methodological lessons generalise beyond this radar:- **Hold one acceptance test fixed** and report failures. A gain error is  constant in dB — correlation, unit slope, and no intensity or range  dependence are non-negotiable.- **Run the null test.** Feed a known 0 dB offset through the estimator; if it  returns −8 dB, the estimator is the finding.- **Choose the unit of comparison deliberately.** More data cannot fix a  variance ceiling; spatial averaging can.- **A result matching your expectation deserves the same scrutiny** as one that  contradicts it. Twice here, a plausible number came from a broken filter.- **Reach for the library routine before writing the geometry yourself.** The  worst error in this analysis — sampling ice and calling it rain — came from a  hand-rolled version of a match that gpm-api already implements.## Acknowledgements**Data.** Radar, disdrometer and sounding data were obtained from theAtmospheric Radiation Measurement (ARM) User Facility, a U.S. Department ofEnergy Office of Science user facility managed by the Biological andEnvironmental Research Program — specifically the Bankhead National Forest(BNF) megasite. GPM DPR 2AKu granules courtesy NASA's Global PrecipitationMeasurement mission via GES DISC and the Precipitation Processing System (PPS).WSR-88D Level II data from NOAA/NCEI, accessed through the Unidata-maintained`unidata-nexrad-level2` archive on AWS.**Software.** This analysis is substantially a composition of four communitypackages, and the credit belongs with them:- **[Py-ART](https://arm-doe.github.io/pyart/)** (Helmus & Collis 2016) — radar  I/O, `GateFilter` gate selection, `columnsect` column extraction,  `subset_radar`, the `kdp_maesaka` retrieval, the `calculate_attenuation_zphi`  correction, the xradar bridge, and `RadarMapDisplay`.- **[gpm-api](https://github.com/ghiggi/gpm_api)** (Ghiggi et al.) — granule  access, the range-slicing helpers, and `gpm.gv.volume_matching` /  `calibration_summary`, which implement the SR–GR method end to end.- **[ACT](https://arm-doe.github.io/ACT/)** (Theisen et al. 2022) — ARM data  discovery and download, facility-location lookup, embedded-QC and Data  Quality Report handling via `act.qc`, and `DistributionDisplay`.- **[cmweather](https://github.com/openradar/cmweather)** — perceptually  considered radar colormaps, and  **[xradar](https://docs.openradarscience.org/projects/xradar/)** — the  cfradial data model the GR side is handed to gpm-api through.With thanks to the wider [Open Radar Science](https://openradarscience.org/)community, whose short courses are where most of this workflow was learned.**Method.** The SR–GR section follows the[ERAD 2026 short-course notebook](https://openradarscience.org/erad2026/notebooks/gpm-api/sr-gr-calibration/)closely, and this notebook is styled after it. The volume-matching method isSchwaller & Morris (2011) as adapted by Warren et al. (2018); RCA followsSilberstein et al. (2008); the disdrometer forward-modelling approach followsthe ARM QUANTS VAP documentation.## References- Helmus, J. J., & Collis, S. M. (2016). The Python ARM Radar Toolkit (Py-ART).  *Journal of Open Research Software*, 4, e25.- Schwaller, M. R., & Morris, K. R. (2011). A ground validation network for the  Global Precipitation Measurement mission. *J. Atmos. Oceanic Technol.*, 28,  301–319.- Silberstein, D. S., Wolff, D. B., Marks, D. A., Atlas, D., & Pippitt, J. L.  (2008). Ground clutter as a monitor of radar stability at Kwajalein, RMI.  *J. Atmos. Oceanic Technol.*, 25, 2037–2045.- Theisen, A., et al. (2022). Atmospheric data Community Toolkit (ACT).  *Journal of Open Source Software*, 7(75), 4495.- Warren, R. A., Protat, A., Siems, S. T., Ramsay, H. A., Louf, V.,  Manton, M. J., & Kane, T. A. (2018). Calibrating ground-based radars against  TRMM and GPM. *J. Atmos. Oceanic Technol.*, 35, 323–346.